In [90]:
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import datasets, transforms

from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR, StepLR
from torch.optim import AdamW

from torch.utils.data import DataLoader

from torch import nn
import torch

from pathlib import Path

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
import numpy as np

In [91]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root='data/train', transform=train_transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = datasets.ImageFolder(root='data/test', transform=test_transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=True)

In [92]:
WARMUP_EPOCHS = 2
MAIN_EPOCHS = 4

model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 4)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

warmup = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS,
)

cosine = CosineAnnealingLR(
    optimizer,
    T_max=MAIN_EPOCHS - WARMUP_EPOCHS,
    eta_min=1e-5,
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup, cosine],
    milestones=[WARMUP_EPOCHS],
)

criterion = nn.CrossEntropyLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

path = Path("Model.pth")

In [93]:
def save():
    print("Saving Model...")

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }

    torch.save(checkpoint, path)

In [94]:
def load():
    if path.exists():
        print(f"Found checkpoint at {path}, Loading model...")

        data = torch.load(path, map_location=device)

        model.load_state_dict(data['model_state_dict'])
        optimizer.load_state_dict(data['optimizer_state_dict'])
        scheduler.load_state_dict(data['scheduler_state_dict'])

    else:
        print(f"No checkpoint found at {path}, Initializing Default Values")

    print(f"============================================================")

In [95]:
def validate():
    model.eval()

    validation_loss = 0
    with torch.no_grad():
        for images, labels in test_loader:

            images, labels = images.to(device), labels.to(device)

            output = model(images)
            loss = criterion(output, labels)

            validation_loss += loss.item()

    validation_loss = validation_loss / len(train_loader)

    model.train()

    return validation_loss

In [96]:
def evaluate():

    model.eval()

    test_loss = 0.0
    all_predictions, all_labels, all_probabilities = [], [], []

    with torch.no_grad():
        for images, labels in test_loader:

            images, labels = images.to(device), labels.to(device)

            output = model(images)

            loss = criterion(output, labels)
            test_loss += loss.item()

            probabilities = torch.softmax(output, dim=1)
            all_probabilities.append(probabilities.cpu().numpy())

            predictions = probabilities.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())

            all_labels.append(labels.cpu().numpy())

    test_loss = test_loss / len(test_loader)

    all_predictions = np.concatenate(all_predictions)
    all_labels = np.concatenate(all_labels)
    all_probabilities = np.concatenate(all_probabilities)

    accuracy = accuracy_score(all_labels, all_predictions)

    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_predictions, average=None, zero_division=0)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_predictions, average="macro", zero_division=0)

    matrix = confusion_matrix(all_labels, all_predictions)

    auc = roc_auc_score(all_labels, all_probabilities, multi_class="ovr", average="macro")

    print(f"==================== Performance Report ====================")

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy:  {(accuracy * 100):.4f}%")

    print()

    print(f"Macro Precision: {precision_macro:.4f}")
    print(f"Macro F1: {f1_macro:.4f}")
    print(f"Macro Recall: {recall_macro:.4f}")

    print()

    print(f"AUC-ROC: {auc:.4f}")

    print()

    print(f"Per-class breakdown:")
    print()
    for i in range(len(precision)):
        print(f"  Class {i}: P={precision[i]:.4f}  R={recall[i]:.4f} F1={f1[i]:.4f}  n={support[i]} ")

    print()

    print(f"Confusion Matrix:\n{matrix}")
    print(f"============================================================")

    model.train()

In [97]:
def train():
    print(f"Initializing Training Session... ")

    load()
    model.train()

    for epoch in range(WARMUP_EPOCHS + MAIN_EPOCHS):

        train_loss = 0
        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)

            output = model(images)

            loss = criterion(output, labels)

            optimizer.zero_grad()
            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        scheduler.step()

        train_loss = train_loss / len(train_loader)
        validation_loss = validate()

        print(f"Epoch: ({epoch + 1}/{WARMUP_EPOCHS + MAIN_EPOCHS})| Train Loss: {train_loss}| Validation Loss: {validation_loss}")

        evaluate()

        user = input("Do You Want To Save The Model? (y/n) ").strip().lower()
        if user == "y":
            save()

        print(f"============================================================")

In [99]:
load()
evaluate()

Found checkpoint at Model.pth, Loading model...
==================== Performance Report ====================
Test Loss: 0.2960
Accuracy:  95.0625%

Macro Precision: 0.9552
Macro F1: 0.9499
Macro Recall: 0.9506

AUC-ROC: 0.9883

Per-class breakdown:

  Class 0: P=1.0000  R=0.8275 F1=0.9056  n=400 
  Class 1: P=0.8744  R=0.9925 F1=0.9297  n=400 
  Class 2: P=0.9540  R=0.9850 F1=0.9692  n=400 
  Class 3: P=0.9925  R=0.9975 F1=0.9950  n=400 

Confusion Matrix:
[[331  50  19   0]
 [  0 397   0   3]
 [  0   6 394   0]
 [  0   1   0 399]]
